# TN0 — Tái lập MobiVital và kiểm chứng pipeline thực nghiệm

Chạy một mạch trên Colab. Không cần notebook nào chạy trước.

## Ba kiểm tra

| | câu hỏi | bắt buộc đạt |
|---|---|---|
| **TN0a** | Tệp lựa chọn kênh tác giả cung cấp có tái hiện điểm công bố **0.819** không? | có |
| **TN0b** | Cùng tệp trọng số `.pth`, pipeline đồ án có chọn đúng 537/537 kênh giống pipeline MobiVital không? | có |
| **TN0c** | Train lại LSTM từ đầu thì đạt mức nào? | không — chỉ tham khảo |

## Luồng thực nghiệm

```
dữ liệu → train → tệp trọng số → chọn kênh → tính điểm → kết luận
```

Chạy hai lần trên cùng bộ dữ liệu:

| | code | các tệp chính |
|---|---|---|
| **pipeline MobiVital** | tác giả cung cấp | `training/autoreg_training.py`, `inference/mobivital_gen.py`, `inference/evaluate.py` |
| **pipeline đồ án** | trong `src/` | `training.py`, `scoring.py`, `results.py` |

Pipeline đồ án tồn tại vì code MobiVital chỉ chạy LSTM — `inference/mobivital_gen.py` dòng 152 ghi cứng `LSTMMultiStep(...)`, không hỗ trợ TCN. TN0 chứng minh nó cho ra đúng kết quả code gốc, rồi mới đổi LSTM sang TCN ở các thực nghiệm sau.

## Hai tệp cùng tên `run_tn0.py` — đọc kỹ chỗ này

Notebook gọi hai tệp tên giống hệt nhau. Chúng làm việc **khác nhau**, phân biệt bằng thư mục:

| lệnh | chạy code của ai | gọi tới |
|---|---|---|
| `scripts/mobivital/run_tn0.py` | **tác giả** | `autoreg_training.py`, `mobivital_gen.py`, `evaluate.py`, `prep_breath_final.py` trong `external/mobivital/` |
| `scripts/run_tn0.py` | **đồ án** | `src/training.py`, `src/scoring.py`, `src/results.py` |

Nhớ một câu: **có `mobivital/` phía trước thì chạy code tác giả, không có thì chạy code đồ án.**

Vì sao để trùng tên: cả hai cùng chạy TN0, chỉ khác chạy bên nào. Đặt tên khác nhau thì mất cái đối xứng đó, nhìn bảng lệnh không thấy ngay hai bên đang làm cùng một việc.

Cụ thể trong notebook này:

```
mục 2   scripts/mobivital/run_tn0.py --case prep    dựng data_final/*.npy
mục 3   scripts/mobivital/run_tn0.py --case a b c   ba kiểm tra, bên tác giả
mục 4   scripts/run_tn0.py           --case a b c   ba kiểm tra, bên đồ án
mục 5   scripts/run_tn0.py           --compare      đối chiếu, in ĐẠT/KHÔNG ĐẠT
```

Chỉ `scripts/run_tn0.py` có `--compare`, vì so sánh là việc của phía đồ án.

## Cách chia việc

```
notebook   →  trình bày luồng và gọi lệnh
scripts/   →  điều khiển thứ tự bước, bắt lỗi, giữ repo tác giả sạch
src/       →  model, train, chọn kênh, tính điểm
```

Mọi kết quả của TN0 nằm chung một thư mục `runs/tn0/` — checkpoint, đường cong loss, bảng lựa chọn kênh, điểm từng buổi ghi, metric. Cuối notebook nén thành `runs/tn0.zip` để tải về.

## 1. Chuẩn bị Colab


Mount Drive để lấy lại dữ liệu đã xử lý ở `DATA_PREPARE.ipynb`, và để cất tệp kết quả ở mục 6.


In [7]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


Tải mã nguồn đồ án về `/content/UWB_RADAR` rồi vào thư mục đó. Sau đó `setup_colab.py` clone MobiVital và **ghim commit `4319731d`** dùng cho mọi số liệu, rồi cài `einops`. Checkpoint giữ lại bằng cách nén ở mục 6 rồi tải về.


In [1]:
# Phải clone repo trước, vì setup_colab.py nằm bên trong chính repo đó.
# %cd phải ở notebook: os.chdir() trong script không đổi thư mục cho ô sau.
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py


/content/UWB_RADAR

thư mục làm việc : /content/UWB_RADAR
commit đồ án     : f504d2c
commit MobiVital : 4319731 (đã ghim)
GPU              : NVIDIA L4, 23034 MiB


## 2. Dữ liệu chung

Một bản CSV duy nhất, đặt trong thư mục MobiVital. Hai pipeline đọc chung bản đó:

```
external/mobivital/dataset/mobivital/tripod/*.csv     1874 tệp
        |
        +-- prep_breath_final.py cua tac gia  ->  external/mobivital/data_final/*.npy
        |
        +-- scripts/make_npz.py cua do an     ->  data/processed/by_user/*.npz
                        |
                  phai khop TUNG BYTE
```

Mỗi lệnh dưới tự bỏ qua nếu đã đủ tệp, nên chạy lại notebook không mất thời gian làm lại. Nhưng một bước đang dở thì phải làm lại từ đầu bước đó.


Tải 5.7 GB từ Zenodo rồi giải nén thành 1874 tệp CSV.


In [2]:
!python scripts/download_dataset.py


Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.36.0-1_amd64.deb ...
Unpacking libaria2-0:amd64 (1.36.0-1) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Setting up libaria2-0:amd64 (1.36.0-1) ...
Setting up aria2 (1.36.0-1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not 

Vá 52 tên tệp lỗi thời để `evaluate.py` mở được, và giấu dữ liệu khỏi git của tác giả.


In [3]:
!python scripts/mobivital/setup_dataset.py


1874 file CSV trong external/mobivital/dataset/mobivital/tripod

1. sao lưu bảng của họ -> runs/tn0/TN0a.txt (537 dòng)
2. external/mobivital/dataset/mobivital/tripod_old_names -> vá 52 tên cũ, 1926 lối tắt
3. thêm dataset/ data_final/ scores*.csv TN0*.txt checkpoints/lstm_retrained* vào .git/info/exclude

KIỂM TRA
   tên trong bảng của họ : 537
   mở không được         : 0  <- phải là 0
   thư mục CSV thật      : 1874  <- phải là 1874

Xong. Giờ chạy được đúng lệnh trong README của MobiVital:
   cd external/mobivital
   python dataset_preparation/prep_breath_final.py
   python -m inference.evaluate -m tripod_mobivital_pre_invert_0.9.txt -d ./dataset/mobivital/tripod_old_names
   python -m inference.mobivital_gen
   python -m training.autoreg_training --model_name lstm_retrained


Chạy `prep_breath_final.py` của tác giả: CSV → `data_final/*.npy` (ABCDEFKL 1289 buổi ghi, GHIJ 537 buổi ghi).


In [4]:
!python scripts/mobivital/run_tn0.py --case prep



$ python dataset_preparation/prep_breath_final.py
100% 1874/1874 [06:41<00:00,  4.67it/s]
100% 1874/1874 [02:49<00:00, 11.07it/s]

total 2578532
drwxr-xr-x  2 root root       4096 Sep  3 20:45 .
drwxr-xr-x 13 root root       4096 Sep  3 20:35 ..
-rw-r--r--  1 root root  776502256 Sep  3 20:45 testing_breath_tripod_data.npy
-rw-r--r--  1 root root 1863894256 Sep  3 20:42 training_breath_tripod_data.npy


Lấy `by_user/` và `windows/` từ Drive nếu `DATA_PREPARE.ipynb` đã cất lên — đỡ khoảng 15 phút. Không có thì hai bước sau tự dựng lại.


In [9]:
!python scripts/restore_processed_data_on_drive.py


by_user   : bung /content/drive/MyDrive/mobivital/by_user.tar ...
            12 tệp
windows   : bung /content/drive/MyDrive/mobivital/windows.tar.gz ...
            dev_cv 8 tệp, final_train có

2.5G	data/processed/by_user
503M	data/processed/windows


Nếu bước trên chưa lấy được thì dựng lại: đọc đúng bộ CSV đó bằng code đồ án, gom theo từng người → `by_user/*.npz`. Đã đủ 12 tệp thì tự bỏ qua.


In [10]:
!python scripts/make_npz.py


đã đủ 12 tệp trong data/processed/by_user — bỏ qua


So hai bên. Phải khớp **từng byte**: ABCDEFKL 1289/1289, GHIJ 537/537. Không khớp thì dừng.


In [11]:
!python scripts/check_data.py


Đối chiếu by_user/*.npz  với  data_final/*.npy

A B C D E F K L
------------------------------------------------------------------
    external/mobivital/data_final/training_breath_tripod_data.npy
    uwb (1289, 1500, 120) complex64 | gt (1289, 1500) float32
    của mình   1289 buổi ghi
    MobiVital  1289 buổi ghi
    khớp TỪNG BYTE 1289/1289

G H I J
------------------------------------------------------------------
    external/mobivital/data_final/testing_breath_tripod_data.npy
    uwb (537, 1500, 120) complex64 | gt (537, 1500) float32
    của mình   537 buổi ghi
    MobiVital  537 buổi ghi
    khớp TỪNG BYTE 537/537

ABCDEFKL  1289/1289 buổi ghi khớp TỪNG BYTE   = training_breath_tripod_data.npy
GHIJ       537/537  buổi ghi khớp TỪNG BYTE   = testing_breath_tripod_data.npy
Từ đây mọi thí nghiệm chỉ đọc by_user/*.npz.


Cắt sẵn cửa sổ train cho pipeline đồ án: 200 mẫu vào → 25 mẫu phải đoán. Đã cắt đủ thì tự bỏ qua.


In [12]:
!python scripts/make_windows.py


cửa sổ đã cắt đủ trong data/processed/windows — bỏ qua


## 3. Pipeline MobiVital

Chạy đúng lệnh trong README của tác giả, không sửa dòng code nào. Script `scripts/mobivital/run_tn0.py` in nguyên văn từng lệnh trước khi chạy, nên đọc output là thấy đủ chuỗi lệnh gốc.

Việc duy nhất script làm thêm: `mobivital_gen.py` ghi tệp lựa chọn kênh đè lên một tệp **có sẵn trong repo tác giả**, nên sau mỗi lần chạy phải chép kết quả ra `runs/tn0/` rồi khôi phục tệp gốc. Cuối mỗi bước script kiểm `git status` của repo đó phải trống.

> Có `mobivital/` phía trước — mục này chạy code **của tác giả**. Xem bảng ở đầu notebook.


**TN0a** — tính điểm từ tệp lựa chọn kênh tác giả cung cấp. Chưa đụng model: kênh khoảng cách và phép biến đổi đã ghi sẵn trong tệp.


In [13]:
!python scripts/mobivital/run_tn0.py --case a



$ python -m inference.evaluate -m tripod_mobivital_pre_invert_0.9.txt -d ./dataset/mobivital/tripod_old_names --save_file scores_TN0a.csv
537it [01:31,  5.86it/s]
0.8194810970476463
   TN0a.txt và scores_TN0a.csv  ->  runs/tn0/


**TN0b** — dùng tệp trọng số tác giả phát hành để chọn kênh trên G H I J. Mỗi buổi ghi: dựng 240 ứng viên (120 kênh khoảng cách × 2 phép biến đổi) → loại các ứng viên bị phát hiện đảo chiều bằng hàm `invert_detector()` của MobiVital → LSTM dự báo → chọn ứng viên có tổng Pearson cao nhất. Bước chọn **không nhìn nhịp thở thật**.


In [23]:
!python scripts/mobivital/run_tn0.py --case b



$ python -m inference.mobivital_gen
100% 1874/1874 [17:25<00:00,  1.79it/s]
0.8221751514440534

$ git checkout -- inference/methods/tripod_mobivital_pre_invert_0.9.txt
   TN0b.txt  537 dòng  ->  runs/tn0/  (đã khôi phục tệp gốc của tác giả)

$ python -m inference.evaluate -m TN0b.txt --save_file scores_TN0b.csv
537it [01:33,  5.71it/s]
0.8221751514440534
   scores_TN0b.csv  ->  runs/tn0/
DỪNG — tệp đã vá còn thay đổi ngoài khối đánh dấu:
+    # === SỬA BỞI ĐỒ ÁN — thêm đúng dòng model.eval() dưới đây ===
+    # Bản gốc không gọi .eval() nên LSTM ở chế độ train; cuDNN cấp thêm vùng nhớ
+    # dự trữ cho backward (15.1 GB cho lô 6708 chuỗi), tràn VRAM mọi GPU Colab.
+    # Model này chỉ có nn.LSTM(dropout=0) + nn.Linear nên train và eval cho
+    # forward giống hệt nhau — vá này không đổi kết quả, chỉ đổi cách xin bộ nhớ.
+    # Chi tiết: scripts/mobivital/patch_eval.py
+    # === HẾT PHẦN SỬA ===


**TN0c** — train lại LSTM từ đầu bằng chính vòng train của tác giả, cấu hình `checkpoints/optimal_params.json`: 20 epoch, Adam lr 1e-4, batch 64, MSE.


In [26]:
!python scripts/mobivital/run_tn0.py --case c



$ python -m training.autoreg_training --model_name lstm_retrained
{'epochs': 20, 'lr': 0.0001, 'batch_size': 64, 'hidden_size': 352, 'history_length': 200, 'future_length': 25, 'num_layers': 2, 'params_file': 'checkpoints/optimal_params.json', 'data_folder': './data_final', 'save_folder': './checkpoints', 'model_name': 'lstm_retrained', 'mode': 'tripod', 'corr_threshold': 0.9, 'top': False, 'device': 0}
  0% 0/20 [00:00<?, ?it/s]Epoch:0 loss: 0.036
 50% 10/20 [12:03<12:03, 72.35s/it]Epoch:10 loss: 0.016
100% 20/20 [24:06<00:00, 72.32s/it]
Finished Training

$ python -m inference.mobivital_gen --model_name lstm_retrained
100% 1874/1874 [17:36<00:00,  1.77it/s]
0.8128385991676149

$ git checkout -- inference/methods/tripod_mobivital_pre_invert_0.9.txt
   TN0c.txt  537 dòng  ->  runs/tn0/  (đã khôi phục tệp gốc của tác giả)

$ python -m inference.evaluate -m TN0c.txt --save_file scores_TN0c.csv
537it [01:34,  5.65it/s]
0.8128385991676149
   scores_TN0c.csv  ->  runs/tn0/

repo MobiVital:

## 4. Pipeline đồ án

Ba kiểm tra như trên, bằng code trong `src/`:

| kiểm tra | pipeline MobiVital | pipeline đồ án |
|---|---|---|
| tính điểm từ tệp lựa chọn kênh | `inference/evaluate.py` | `scoring.score_from_txt` |
| chọn kênh từ tệp trọng số | `inference/mobivital_gen.py` | `scoring.score_all` |
| train lại LSTM | `training/autoreg_training.py` | `training.train` |

Mỗi `--case` thêm đúng một bộ phận, nên bộ phận nào sai thì lộ ra ở đúng kiểm tra đó.

> Không có `mobivital/` phía trước — mục này chạy code **của đồ án** trong `src/`. Xem bảng ở đầu notebook.


**TN0a** — chỉ dùng hàm tính điểm, chưa chạy model.


In [28]:
!python scripts/run_tn0.py --case a


thiết bị: cuda NVIDIA L4

TN0a — tính điểm từ runs/tn0/TN0a.txt (không chạy model)
ours_a    537 buổi ghi   điểm trung bình 0.8194810970
          -> runs/tn0/scores_ours_a.csv


**TN0b** — thêm bộ chọn kênh, vẫn dùng tệp trọng số tác giả phát hành.


In [29]:
!python scripts/run_tn0.py --case b


thiết bị: cuda NVIDIA L4

TN0b — chọn kênh bằng external/mobivital/checkpoints/lstm_pred_tripod_0.9.pth
ours_b    537 buổi ghi   điểm trung bình 0.8221751514
          -> runs/tn0/scores_ours_b.csv
          -> runs/tn0/ours_b.txt


**TN0c** — thêm vòng train, cùng cấu hình MobiVital công bố.


In [30]:
!python scripts/run_tn0.py --case c


thiết bị: cuda NVIDIA L4

TN0c — train lại LSTM, cấu hình MobiVital công bố:
       20 epoch, Adam lr 0.0001, batch 64, MSE
       292708 cửa sổ train
epoch  0  mse 0.03749  pearson 0.5228   1.2 phút
epoch  1  mse 0.02386  pearson 0.5806   2.5 phút
epoch  2  mse 0.02181  pearson 0.6023   3.7 phút
epoch  3  mse 0.02047  pearson 0.6164   4.9 phút
epoch  4  mse 0.01952  pearson 0.6251   6.1 phút
epoch  5  mse 0.01877  pearson 0.6328   7.4 phút
epoch  6  mse 0.01808  pearson 0.6394   8.6 phút
epoch  7  mse 0.01748  pearson 0.6443   9.8 phút
epoch  8  mse 0.01699  pearson 0.6488   11.0 phút
epoch  9  mse 0.01652  pearson 0.6536   12.3 phút
epoch 10  mse 0.01615  pearson 0.6563   13.5 phút
epoch 11  mse 0.01578  pearson 0.6601   14.7 phút
epoch 12  mse 0.01549  pearson 0.6641   15.9 phút
epoch 13  mse 0.01524  pearson 0.6663   17.2 phút
epoch 14  mse 0.01497  pearson 0.6688   18.4 phút
epoch 15  mse 0.01473  pearson 0.6721   19.6 phút
epoch 16  mse 0.01450  pearson 0.6737   20.8 phút
epoch 1

## 5. So sánh hai pipeline

Lệnh dưới kiểm bốn điều và trả mã lỗi khác 0 nếu có điều bắt buộc không đạt:

1. **TN0a tái hiện bài báo** — điểm pipeline MobiVital lệch `0.819` dưới `0.001`.
2. **TN0a hai pipeline khớp** — chênh lệch trên **từng** buổi ghi dưới `1e-9`, không chỉ so điểm trung bình.
3. **TN0b hai pipeline khớp** — 537/537 buổi ghi chọn cùng kênh, chênh lệch từng buổi dưới `1e-9`.
4. **Repo MobiVital sạch** — `git status --porcelain` trống.

TN0c chỉ tham khảo vì đây là hai lần train độc lập; không yêu cầu trọng số và điểm số giống tuyệt đối.


In [33]:
!python scripts/run_tn0.py --compare



Kiểm tra                     MobiVital     Pipeline đồ án  Kết luận
----------------------------------------------------------------------------
TN0a  điểm từ TXT tác giả    0.819481      0.819481        ĐẠT
TN0b  cùng tệp trọng số      0.822175      0.822175        537/537 kênh — ĐẠT
TN0c  train lại LSTM         0.812839      0.822642        thông tin tham khảo
----------------------------------------------------------------------------
TN0a  so với bài báo 0.819     : lệch 0.00048  ĐẠT
TN0a  chênh lệch lớn nhất trên 537 buổi ghi : 0.00e+00
TN0b  chênh lệch lớn nhất trên 537 buổi ghi : 0.00e+00
repo MobiVital                : ĐẠT — chỉ inference/mobivital_gen.py bị sửa, 8 dòng thêm, toàn bộ trong khối đánh dấu (đúng một dòng lệnh model.eval())

bảng trên đã ghi ra runs/tn0/compare.csv

TN0 ĐẠT — pipeline đồ án cho ra đúng kết quả pipeline MobiVital.


## 6. Lưu kết quả

Nén cả thư mục `runs/tn0/` thành `runs/tn0.zip`, và chép sang Drive nếu đã mount. Giải nén lại bằng `unzip tn0.zip -d runs/`.


In [34]:
!python scripts/save_results.py tn0


runs/tn0/  ->  runs/tn0.zip   (5.6 MB)
   0 dòng metric trong summary.csv

Bên trong:
        0  2026-09-04 00:57   tn0/
        0  2026-09-04 00:45   tn0/ours_c/
      153  2026-09-04 00:57   tn0/README.txt
    21078  2026-09-04 00:19   tn0/ours_b.txt
    25802  2026-09-03 21:00   tn0/scores_TN0a.csv
    25774  2026-09-03 23:22   tn0/scores_TN0b.csv
    34540  2026-09-04 00:55   tn0/scores_ours_c.csv
    21078  2026-09-03 23:20   tn0/TN0b.txt
    21060  2026-09-04 00:07   tn0/TN0c.txt
    21078  2026-09-03 20:59   tn0/TN0a.txt
    34542  2026-09-04 00:19   tn0/scores_ours_b.csv
    25776  2026-09-04 00:08   tn0/scores_TN0c.csv
      657  2026-09-04 00:56   tn0/compare.csv
    21083  2026-09-04 00:55   tn0/ours_c.txt
    32944  2026-09-04 00:10   tn0/scores_ours_a.csv
     1493  2026-09-04 00:45   tn0/ours_c/curve.csv
  6013519  2026-09-04 00:45   tn0/ours_c/final.pth
---------                     -------
  6300577                     17 files


đã chép sang /content/drive/MyDrive/mobi